# 유사한 단어 찾기 게임

1. 사전 학습된 모델 또는 적절한 데이터셋을 찾는다.
2. 워드 임베딩 모델을 학습시킨다.
3. 단어 유사도가 0.8 이상인 A, B를 랜덤 추출한다.
4. A, B와 대응되는 C를 추출한다.
5. D를 입력 받는다.

=>
A:B = C:D 관계에 대응하는 D를 찾는 게임을 만든다.
ex) A: 산, B: 바다, C: 나무, D: 물

**<출력 예시>**

관계 [ 수긍 : 추락 = 대사관 : ? ]<br>
모델이 예측한 가장 적합한 단어: 잠입<br>
당신의 답변과 모델 예측의 유사도: 0.34<br>
아쉽네요. 더 생각해보세요.

In [1]:
import pandas as pd

splits = {'train': 'dp/train-00000-of-00001.parquet', 'validation': 'dp/validation-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/klue/klue/" + splits["train"])

In [2]:
df = df['sentence']

In [3]:
from lxml import etree
import re
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords

In [4]:
from konlpy.tag import Okt
from tqdm import tqdm
import re

okt = Okt()

# 기존 불용어 + 확장
ko_stopwords = [
    "은","는","이","가","을","를","과","와","들","도","부터","까지","에","나","너","그","걔","얘",
    "다","하다","되다","같다","있다",
    # 의미 없는 명사성 단어 추가
    "대해","위해","통해","정도","부분","경우","사실","때문","이번","이번에","관련"
]

preprocessed_data = []

for sentence in tqdm(df):
    sentence = re.sub(r"[a-zA-Z]", " ", sentence)   # 영문 제거
    sentence = re.sub(r"\d+", " ", sentence)        # 숫자 제거
    sentence = re.sub(r"[^가-힣\s]", " ", sentence) # 특수문자 제거

    # 품사 태깅
    morphs = okt.pos(sentence, stem=True)

    # 명사만 추출 + 불용어 제거 + 길이 2 이상
    tokens = [
        word for word, tag in morphs
        if tag in ["Noun", "ProperNoun"]
        and word not in ko_stopwords
        and len(word) > 1
    ]

    preprocessed_data.append(tokens)


100%|██████████| 10000/10000 [00:16<00:00, 615.39it/s]


In [5]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=preprocessed_data,
    vector_size=200,  
    sg=1,             
    window=5,
    min_count=3,      
    workers=4,
    epochs=15       
)

In [6]:
import pandas as pd

pd.DataFrame(model.wv.vectors, index=model.wv.index_to_key).head(10)

,0,1,2,3,4,5,6,7,8,9,...,190,191,192,193,194,195,196,197,198,199
숙소,0.147324,-0.138676,-0.095218,0.310055,0.011295,-0.144005,-0.054127,0.666137,-0.248075,0.224795,...,0.110834,-0.185477,-0.028763,-0.395938,0.180298,0.223501,0.203262,-0.323095,-0.090035,-0.359033
위치,0.117976,0.088541,-0.318459,0.411171,0.058163,-0.134819,0.000491,0.309982,-0.242351,0.455658,...,-0.076199,-0.229357,-0.163383,-0.491804,0.369244,0.188462,0.214319,-0.290374,0.096176,-0.325293
호스트,-0.101900,-0.142335,-0.219352,0.429529,-0.070107,0.270262,0.000038,0.683579,-0.411444,0.295621,...,0.197128,0.041119,-0.016104,-0.474936,0.373448,0.095200,-0.010736,-0.548172,0.123608,-0.300735
지난,0.034169,-0.048733,0.031552,-0.203281,0.173584,-0.047158,0.002213,-0.000645,-0.266294,0.019147,...,0.003833,0.005031,-0.272365,0.127246,0.178515,0.147309,-0.226610,-0.062374,-0.004572,0.062359
정말,-0.081169,-0.058418,-0.098022,0.357679,-0.063584,0.133991,0.053767,0.612159,-0.255054,0.294986,...,0.090123,-0.099454,-0.041016,-0.456816,0.365094,0.114846,0.022483,-0.359350,-0.022630,-0.244812
시간,0.138135,-0.020175,0.290121,-0.291347,-0.110528,-0.358169,-0.179096,0.259061,-0.204624,0.117272,...,0.201922,-0.123447,-0.151449,-0.146250,0.572109,0.022823,-0.322423,-0.299712,-0.051930,-0.010031
여행,0.275596,-0.138790,-0.014733,0.139654,0.240957,0.194433,0.034745,0.733443,-0.107988,0.010971,...,0.059014,-0.167380,-0.211493,-0.280525,-0.029457,0.054328,0.262895,-0.368129,0.165902,0.012894
사진,0.087285,0.378273,-0.152466,0.013851,0.105157,-0.061277,0.306350,0.207935,-0.202998,0.068222,...,-0.172229,0.165693,-0.335614,-0.335601,0.097660,0.170626,0.197587,0.039583,-0.000277,0.020224
매우,0.024190,-0.170515,-0.044131,0.343956,-0.019081,0.017726,-0.064288,0.436362,-0.093639,0.330765,...,0.032747,-0.120793,-0.042701,-0.476037,0.330154,0.177432,-0.012250,-0.285531,0.069096,-0.291477
한국,0.124129,-0.141788,-0.231112,-0.113478,0.389041,0.130170,0.015989,0.310742,-0.356512,0.184093,...,-0.156003,-0.005970,-0.171076,0.208581,0.062200,0.189827,-0.194974,0.113005,0.075085,0.396431


In [7]:
# model : Word2Vec
model.wv.most_similar('남자')

[('무대', 0.8915528059005737),
 ('여자', 0.890527606010437),
 ('응원', 0.8795218467712402),
 ('댄스', 0.8722191452980042),
 ('주인공', 0.8699138164520264),
 ('스타', 0.8658517599105835),
 ('마치', 0.865611732006073),
 ('린지', 0.8637828826904297),
 ('인스타그램', 0.8627666234970093),
 ('배경', 0.8616982102394104)]

In [9]:
import random

kv = model.wv

def play():
    vocab = list(kv.key_to_index.keys())

    # A, B 두 단어 랜덤 선택한다.
    A, B = random.sample(vocab, 2)

    # A와 가장 유사한 단어 C 선택한다.
    try:
        C = kv.most_similar(A, topn=1)[0][0]
    except KeyError:
        print("해당 단어로는 유사도 계산 불가. 다시 실행하세요.")
        return

    # 모델 예측: A:B = C:?
    try:
        pred = kv.most_similar(positive=[B, C], negative=[A], topn=1)[0][0]
    except KeyError:
        print("관계 계산 불가. 다시 실행하세요.")
        return

    print(f"관계 [ {A} : {B} = {C} : ? ]")
    print(f"모델이 예측한 가장 적합한 단어: {pred}")

    user = input("D를 입력하세요: ").strip()
    print(f"당신이 선택한 단어: {user}")

    if user in kv and pred in kv:
        sim = kv.similarity(user, pred)
        print(f"당신의 답변과 모델 예측의 유사도: {sim:.2f}")
        if sim < 0.7:
            print("아쉽네요. 더 생각해보세요.")
    else:
        print("사전에 없는 단어라 유사도 계산 불가")

In [13]:
play()

관계 [ 마트 : 금메달 = 편의점 : ? ]
모델이 예측한 가장 적합한 단어: 랠리
당신이 선택한 단어: 은메달
당신의 답변과 모델 예측의 유사도: 0.97
